# Loading script for Green stay hotels in DWHPFX schema

In [1]:
import requests
import json
import pandas as pd
import configparser
import pyexasol

def GreenStayExtract():
    try:
       ### Initialize the dataframe
        final_df = pd.DataFrame()
        ### API link
        url = "https://hotel-audit-webservice-stage-q3rx4x4ohq-ew.a.run.app/v1/audits/report/green?secretKey=3AwuZKz9nH&page=1&size=100"
        parsed = json.loads(requests.get(url).text)

        ## Get the number of pages through -> #int(parsed['total_pages'])
        for x in range(int(parsed['total_pages'])):
            url_iter = "https://hotel-audit-webservice-stage-q3rx4x4ohq-ew.a.run.app/v1/audits/report/green?secretKey=3AwuZKz9nH&page="+str(x+1)+"&size=100"
            response = requests.get(url_iter)

            if response.status_code == 200:    
                data = response.text
                parsed = json.loads(data)
                df = pd.DataFrame(parsed.get('results'))
                final_df = final_df.append(df, ignore_index=True)
            else: 
                print("Request failed page: {} ".format(x))
        final_df['created_date']= final_df['created_date'].astype(str).str[:-6]
        final_df['updated_date']= final_df['updated_date'].astype(str).str[:-6]
        final_df['created_date']= pd.to_datetime(final_df['created_date'], utc=False)
        final_df['updated_date']= pd.to_datetime(final_df['updated_date'], utc=False)
        final_df = final_df[final_df.hkey.notnull()]
        final_df = final_df[['hkey','created_date','updated_date','report_year','kilogramCarbonPOC','literWaterPOC',
                             'kilogramWastePOC','carbonClass','waterClass','wasteClass','greenClass','type','status']]
        final_df.rename(columns = {'hkey':'HOTEL_ID', 'created_date':'CREATED_DATE','updated_date':'UPDATED_DATE',
                                   'report_year': 'REPORTING_YEAR','kilogramCarbonPOC': 'CARBON_KGS','literWaterPOC':'WATER_LTS',
                                   'kilogramWastePOC': 'WASTE_KGS','carbonClass':'CARBON_CLASS','waterClass':'WATER_CLASS',
                                   'wasteClass':'WASTE_CLASS','greenClass':'GREEN_CLASS','type':'GREEN_INSPECTION_TYPE',
                                   'status':'GREEN_INSPECTION_STATUS'
                                  }, inplace = True)
        harmonized_df = final_df.sort_values('UPDATED_DATE').groupby('HOTEL_ID').tail(1)
        harmonized_df['CREATED_DATE']= pd.to_datetime(final_df['CREATED_DATE'], format='%Y%m%d', errors='ignore').dt.date
        harmonized_df['UPDATED_DATE']= pd.to_datetime(final_df['UPDATED_DATE'], format='%Y%m%d', errors='ignore').dt.date
        harmonized_df['GREEN_INSPECTION_TYPE'] = harmonized_df['GREEN_INSPECTION_TYPE'].apply(lambda x: x.upper())
        print(f'Number of records having HOTEL_IDs {final_df.HOTEL_ID.nunique()}.')
        print(f'Number of records with duplicate HOTEL_IDs: {final_df.duplicated(subset="HOTEL_ID", keep="first").sum()}.')
    except Exception as e:
        print('Failed in function GreenScore - ')
        raise e
    return harmonized_df


def GreenStayLoad(df): 
    harmonized_df = df
    try:
        #Location of the ini file
        config = configparser.ConfigParser()
        ## Config location CHANGE
        config.read('C:\\Users\\USER\\.spyder-py3\\pfxDET.ini')
        dsn=config['pfxDET']['dsn']
        user=config['pfxDET']['user']
        pwd=config['pfxDET']['pwd']
        schema=config['pfxDET']['schema']
        # Exasol connection
        connect = pyexasol.connect(dsn=dsn, user=user, password=pwd, schema=schema)
        connect.execute("TRUNCATE TABLE DWHPFX.GREEN_STAY_HOTELS")
        connect.import_from_pandas(harmonized_df, table = ('DWHPFX','GREEN_STAY_HOTELS'))
        stmt = connect.last_statement()
        print(f'Number of records inserted  {stmt.rowcount()}.')
    except Exception as e:
        print('Failed in function GreenStayLoad - ')
        raise e

### Function call

In [2]:
df = GreenStayExtract()
GreenStayLoad(df)

### Test results in excel

In [106]:
from datetime import date
df.to_excel('C:\\Users\\USER\\Documents\\misc\\'+str(date.today())+'_test.xlsx', 
              sheet_name='Sheet1', 
              header=True,
              encoding='utf-8',
              index=False,
              freeze_panes=(1,0) )